# Build APK Kalkulator Saham (Python → Android)

1. Buka menu **Runtime → Run all** (Runtime → Jalankan semua).
2. Tunggu sekitar **20–40 menit**. Build pertama mengunduh Android SDK/NDK dan meng-compile Python untuk Android. Sel ke-3 menampilkan progres tiap menit.
3. Di akhir, APK diunduh otomatis. Di HP unduhan kadang tidak mulai; sel terakhir menyimpan APK ke **Google Drive** Anda (izinkan akses saat diminta).

Kalau build gagal, salin 80 baris log yang muncul dan kirim ke Claude.

In [ ]:
# 1) Pasang alat build
import os, sys
print("Python", sys.version.split()[0])
!apt-get update -qq
!apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf automake libtool pkg-config zlib1g-dev libncurses-dev cmake libffi-dev libssl-dev > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
cython = "cython==0.29.37" if sys.version_info < (3, 13) else "cython==3.0.11"
!pip install -q --upgrade buildozer "$cython" virtualenv setuptools
!java -version 2>&1 | head -1
!buildozer --version

In [ ]:
# 2) Ambil kode aplikasi dari GitHub
%cd /content
!rm -rf kalkulator
!git clone -q --depth 1 https://github.com/muhamadwijaya/kalkulator
%cd /content/kalkulator
!python -c "import calc; print('Uji rumus OK:', calc.hitung(10_000_000, 2, 1000, 950, 2)['lot'], 'lot')"

In [ ]:
# 3) Build APK (lama, progres muncul tiap menit)
import subprocess, time
t0 = time.time()
with open("build.log", "w") as log:
    p = subprocess.Popen("yes | buildozer -v android debug", shell=True,
                         stdout=log, stderr=subprocess.STDOUT, cwd="/content/kalkulator")
    while p.poll() is None:
        time.sleep(60)
        try:
            baris = [b for b in open("build.log", errors="replace").read().splitlines() if b.strip()]
            akhir = baris[-1][:140] if baris else ""
        except Exception:
            akhir = ""
        print(f"[{int((time.time() - t0) / 60)} menit] {akhir}", flush=True)
print(f"Selesai dalam {(time.time() - t0) / 60:.1f} menit, kode keluar {p.returncode}")
if p.returncode != 0:
    !tail -n 80 build.log
    raise RuntimeError("Build gagal. Salin 80 baris log di atas lalu kirim ke Claude.")
!ls -la bin/

In [ ]:
# 4) Unduh APK
import glob
from google.colab import files
apk = sorted(glob.glob("/content/kalkulator/bin/*.apk"))[-1]
print("APK:", os.path.basename(apk), f"({os.path.getsize(apk) / 1e6:.1f} MB)")
files.download(apk)

In [ ]:
# 5) Simpan juga ke Google Drive (berguna kalau unduhan di HP tidak mulai)
import shutil
from google.colab import drive
drive.mount("/content/drive")
shutil.copy(apk, "/content/drive/MyDrive/")
print("Tersimpan di Google Drive: My Drive/" + os.path.basename(apk))